In [1]:
import time

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt
        
    if c > 0: successors.append(("Trái", swap(mt, pos, pos - 1)))
    if c < 2: successors.append(("Phải", swap(mt, pos, pos + 1)))
    if r > 0: successors.append(("Lên", swap(mt, pos, pos - 3)))
    if r < 2: successors.append(("Xuống", swap(mt, pos, pos + 3)))
    return successors

def and_or_graph_search(start_state, goal_state, limit):
    nodes_generated = 1
    
    def get_actions(state):
        pos = state.index(0)
        r, c = pos // 3, pos % 3
        actions = []
        if c > 0: actions.append("Trái")
        if c < 2: actions.append("Phải")
        if r > 0: actions.append("Lên")
        if r < 2: actions.append("Xuống")
        return actions

    def get_result(state, action):
        pos = state.index(0)
        new_mt = list(state)
        if action == "Trái":
            new_mt[pos], new_mt[pos - 1] = new_mt[pos - 1], new_mt[pos]
        elif action == "Phải":
            new_mt[pos], new_mt[pos + 1] = new_mt[pos + 1], new_mt[pos]
        elif action == "Lên":
            new_mt[pos], new_mt[pos - 3] = new_mt[pos - 3], new_mt[pos]
        elif action == "Xuống":
            new_mt[pos], new_mt[pos + 3] = new_mt[pos + 3], new_mt[pos]
        return [new_mt]

    def or_search(state, path):
        nonlocal nodes_generated
        if state == goal_state:
            return []
        if state in path:
            return "failure"
        if len(path) >= limit:
            return "failure"
            
        for action in get_actions(state):
            result_states = get_result(state, action)
            for r_state in result_states:
                nodes_generated += 1
            plan = and_search(result_states, path + [state])
            if plan != "failure":
                return [action, plan]
        return "failure"

    def and_search(states, path):
        plans = {}
        for s in states:
            plan_s = or_search(s, path)
            if plan_s == "failure":
                return "failure"
            plans[tuple(s)] = plan_s
        return plans

    plan = or_search(start_state, [])
    return plan, nodes_generated

def plan_to_path(plan):
    if plan == "failure":
        return None
    path = []
    current_plan = plan
    while current_plan:
        action = current_plan[0]
        plans = current_plan[1]
        if not plans:
            break
        child_state_tuple = list(plans.keys())[0]
        child_state = list(child_state_tuple)
        path.append((action, child_state))
        current_plan = plans[child_state_tuple]
    return path
